In [35]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import math
import time
from datetime import timedelta
from tabulate import tabulate
from xgboost import XGBClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, make_scorer, roc_auc_score, accuracy_score
from sklearn.utils.class_weight import compute_sample_weight 
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")

# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Carico csv

In [36]:
# Carico SOLO i dataset radiomici
df_duke = pd.read_csv(FILE_PATH / "duke_lesions_radiomic_medsam.csv")
df_ambl = pd.read_csv(FILE_PATH / "ambl_lesions_radiomic_medsam.csv")

print("DUKE shape:", df_duke.shape)
print("AMBL shape:", df_ambl.shape)


DUKE shape: (291, 109)
AMBL shape: (82, 111)


# Definisco il target di interesse

In [37]:
target_row = {
    "PR" : ["PR", "PR [SII]"]
}

# Cerco a colonna PR nei csv

In [38]:
def get_target_column (df, target_aliases):
    for col in target_aliases:
        if col in df.columns:
            return col
    raise ValueError(f"Nessuna colonna target trovata tra: {target_aliases}")

# Binarizzo ambl

In [39]:
df_ambl["PR_binario"] = (
        pd.to_numeric(df_ambl["PR [SII]"], errors="coerce") >= 1
    ).astype(int)

# Applico la funzione

In [40]:
target_duke = get_target_column(df_duke, target_row["PR"])
target_ambl = "PR_binario"

print("Target PR DUKE:", target_duke)
print("Target PR AMBL:", target_ambl)

Target PR DUKE: PR
Target PR AMBL: PR_binario


# Controllo delle classi

In [41]:
print("\nDistribuzione PR DUKE:")
print(df_duke[target_duke].value_counts(dropna=False))

print("\nDistribuzione PR AMBL:")
print(df_ambl[target_ambl].value_counts(dropna=False))



Distribuzione PR DUKE:
PR
0    157
1    134
Name: count, dtype: int64

Distribuzione PR AMBL:
PR_binario
0    46
1    36
Name: count, dtype: int64


# Preparo le feature

In [42]:
def prepare_features(df, target_col):
    features_to_drop = [
        "Patient ID", "lesion idx", "tumor/benign",
        "GRADE", "isTN", "Breast", target_col
    ]

    X = df.drop(columns=features_to_drop, errors="ignore")
    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.fillna(X.mean(numeric_only=True))
    X.columns = [re.sub(r"\[|\]|<", "", c) for c in X.columns]

    return X

# Split duke (Train e Testing)

In [43]:
X_full = prepare_features(df_duke, target_duke)
y_full = df_duke[target_duke]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, val_idx = next(skf.split(X_full, y_full))

train_original = df_duke.iloc[train_idx].reset_index(drop=True)
val_internal   = df_duke.iloc[val_idx].reset_index(drop=True)

print("Train DUKE:", train_original.shape)
print("Val DUKE:", val_internal.shape)

Train DUKE: (232, 109)
Val DUKE: (59, 109)


# Classi uguali (solo in fase di Train)

In [44]:
class_counts = train_original[target_duke].value_counts()
n_min = class_counts.min()

train_balanced = (
    train_original
    .groupby(target_duke, group_keys=False)
    .apply(lambda x: x.sample(n=n_min, random_state=42))
    .reset_index(drop=True)
)

print("Distribuzione PR dopo bilanciamento:")
print(train_balanced[target_duke].value_counts())

Distribuzione PR dopo bilanciamento:
PR
0    107
1    107
Name: count, dtype: int64


/var/folders/fp/7t9yf86d2d38gysngbl33_y40000gn/T/ipykernel_57009/1953424125.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_original


# Feature finali

In [45]:
X_train = prepare_features(train_balanced, target_duke)
FEATURE_COLUMNS = X_train.columns.tolist()
y_train = train_balanced[target_duke]

X_val = prepare_features(val_internal, target_duke)
X_val = X_val.reindex(columns=FEATURE_COLUMNS)


X_test = prepare_features(df_ambl, target_ambl)
X_test = X_test.reindex(columns=FEATURE_COLUMNS)

# Definisco il modello XGBoost

In [46]:
model = XGBClassifier(
        random_state=42,
        n_jobs=1,
        objective='binary:logistic',
        eval_metric='logloss',
        tree_method='hist',
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=3,
        reg_alpha=0,
        reg_lambda=1,
        scale_pos_weight=1,
    )
model.fit(X_train, y_train);

# Validazione Duke

In [47]:
y_val = val_internal[target_duke]

y_val_pred = model.predict(X_val)
y_val_proba = model.predict_proba(X_val)[:, 1]

print("\nDUKE – Validation")
print(f"F1 : {f1_score(y_val, y_val_pred):.3f}")
print(f"AUC: {roc_auc_score(y_val, y_val_proba):.3f}")
print(f"ACC: {accuracy_score(y_val, y_val_pred):.3f}")



DUKE – Validation
F1 : 0.793
AUC: 0.843
ACC: 0.797


# Test eseterno con ambl

In [48]:
y_test = df_ambl["PR_binario"]

y_test_pred = model.predict(X_test)
y_test_proba = model.predict_proba(X_test)[:, 1]

print("\nAMBL – External Test")
print(f"F1 : {f1_score(y_test, y_test_pred):.3f}")
print(f"AUC: {roc_auc_score(y_test, y_test_proba):.3f}")
print(f"ACC: {accuracy_score(y_test, y_test_pred):.3f}")



AMBL – External Test
F1 : 0.617
AUC: 0.608
ACC: 0.500
